In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

from sklearn.pipeline import Pipeline

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

import joblib

In [2]:
df = pd.read_csv("my_dataset.csv")

In [3]:
df["priority"].value_counts(dropna=False)

,count
priority,
High,5030
Low,3141
NaN,2


In [4]:
features = [
    "event_type",
    "event_cause",
    "veh_type",
    "latitude",
    "longitude"
]

target = "priority"

df = df[features + [target]]

In [5]:
df["veh_type"] = df["veh_type"].fillna("unknown")

df = df.dropna(subset=["priority"])

In [6]:
df["priority"] = df["priority"].map({
    "Low": 0,
    "High": 1
})

In [7]:
df["priority"].value_counts()

,count
priority,
1,5030
0,3141


In [8]:
X = df[features]

y = df["priority"]

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [10]:
categorical_features = [
    "event_type",
    "event_cause",
    "veh_type"
]

numerical_features = [
    "latitude",
    "longitude"
]

In [11]:
preprocessor = ColumnTransformer(
    [
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "num",
            "passthrough",
            numerical_features
        )
    ]
)

In [12]:
model = Pipeline([
    ("preprocessor", preprocessor),

    (
        "classifier",
        RandomForestClassifier(
            n_estimators=300,
            max_depth=15,
            random_state=42
        )
    )
])

In [13]:
model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['event_type', 'event_cause',
                                                   'veh_type']),
                                                 ('num', 'passthrough',
                                                  ['latitude', 'longitude'])])),
                ('classifier',
                 RandomForestClassifier(max_depth=15, n_estimators=300,
                                        random_state=42))])

In [14]:
predictions = model.predict(X_test)

In [15]:
accuracy = accuracy_score(
    y_test,
    predictions
)

print("Accuracy:", accuracy)

Accuracy: 0.8788990825688073


In [16]:
print(
    classification_report(
        y_test,
        predictions
    )
)

              precision    recall  f1-score   support

           0       0.94      0.73      0.82       629
           1       0.85      0.97      0.91      1006

    accuracy                           0.88      1635
   macro avg       0.90      0.85      0.87      1635
weighted avg       0.89      0.88      0.88      1635



In [17]:
print(
    confusion_matrix(
        y_test,
        predictions
    )
)

[[459 170]
 [ 28 978]]


In [18]:
probs = model.predict_proba(X_test)[:,1]

roc_auc = roc_auc_score(
    y_test,
    probs
)

print("ROC AUC:", roc_auc)

ROC AUC: 0.9547042071892967


In [19]:
feature_names = model.named_steps[
    "preprocessor"
].get_feature_names_out()

importances = model.named_steps[
    "classifier"
].feature_importances_

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
})

importance_df.sort_values(
    "importance",
    ascending=False
).head(20)

,feature,importance
31,num__longitude,0.526652
30,num__latitude,0.380475
15,cat__event_cause_tree_fall,0.016013
20,cat__veh_type_bmtc_bus,0.008426
16,cat__event_cause_vehicle_breakdown,0.007856
29,cat__veh_type_unknown,0.006756
4,cat__event_cause_accident,0.006697
26,cat__veh_type_private_car,0.004167
10,cat__event_cause_procession,0.003952
6,cat__event_cause_construction,0.003724


In [20]:
joblib.dump(
    model,
    "priority_model.pkl"
)

['priority_model.pkl']

In [21]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, predictions)

print("Accuracy:", accuracy)

Accuracy: 0.8788990825688073


In [22]:
from sklearn.metrics import classification_report

print(classification_report(y_test, predictions))

              precision    recall  f1-score   support

           0       0.94      0.73      0.82       629
           1       0.85      0.97      0.91      1006

    accuracy                           0.88      1635
   macro avg       0.90      0.85      0.87      1635
weighted avg       0.89      0.88      0.88      1635



In [23]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, predictions)

print(cm)

[[459 170]
 [ 28 978]]


In [24]:
from sklearn.metrics import roc_auc_score

probs = model.predict_proba(X_test)[:, 1]

roc_auc = roc_auc_score(y_test, probs)

print("ROC AUC:", roc_auc)

ROC AUC: 0.9547042071892967


In [25]:
feature_names = model.named_steps[
    "preprocessor"
].get_feature_names_out()

importances = model.named_steps[
    "classifier"
].feature_importances_

import pandas as pd

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
})

importance_df = importance_df.sort_values(
    "importance",
    ascending=False
)

importance_df.head(20)

,feature,importance
31,num__longitude,0.526652
30,num__latitude,0.380475
15,cat__event_cause_tree_fall,0.016013
20,cat__veh_type_bmtc_bus,0.008426
16,cat__event_cause_vehicle_breakdown,0.007856
29,cat__veh_type_unknown,0.006756
4,cat__event_cause_accident,0.006697
26,cat__veh_type_private_car,0.004167
10,cat__event_cause_procession,0.003952
6,cat__event_cause_construction,0.003724


In [26]:
import joblib

joblib.dump(
    model,
    "priority_model.pkl"
)

print("Priority model saved")

Priority model saved


In [27]:
from google.colab import files

files.download("priority_model.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>